In [0]:
%sql
---- Creating new catalog, schema -----
use catalog sql_youtube_practise;
create schema if not exists pyspark;
use pyspark;
show current schema;

catalog,namespace
sql_youtube_practise,pyspark


In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
data = [
    ("India", "SL", "India"),
    ("SL", "Aus", "Aus"),
    ("SA", "Eng", "Eng"),
    ("Eng", "NZ", "NZ"),
    ("Aus", "India", "India")
]
columns = ["Team_1", "Team_2", "Winner"]
icc_world_cup_df = spark.createDataFrame(data, columns)
icc_world_cup_df.write.mode("overwrite").saveAsTable("icc_world_cup")
icc_world_cup_df.display()

Team_1,Team_2,Winner
India,SL,India
SL,Aus,Aus
SA,Eng,Eng
Eng,NZ,NZ
Aus,India,India


In [0]:
from pyspark.sql.functions import *
team1_df = icc_world_cup_df.select(
    col("Team_1").alias("team_name"),
    when(col("Winner") == col("Team_1"), 1).otherwise(0).alias("wins")
)
team2_df = icc_world_cup_df.select(
    col("Team_2").alias("team_name"),
    when(col("Winner") == col("Team_2"), 1).otherwise(0).alias("wins")
)
team = team1_df.unionAll(team2_df) \
        .groupBy("team_name") \
        .agg(
            count("*").alias("total_matches_played"),
            sum("Wins").alias("Wins")
        ) \
        .withColumn(
            "Losses",
            col("total_matches_played") - col("Wins")
        ) \
        .orderBy(col("Wins").desc())
team.display()

team_name,total_matches_played,Wins,Losses
India,2,2,0
Eng,2,1,1
Aus,2,1,1
NZ,1,1,0
SL,2,0,2
SA,1,0,1
